In [2]:
import requests
print("imported requests module")

imported requests module


In [24]:
url = "https://v2.jokeapi.dev/joke/Dark"

headers = {

}
params={"limit": 100, "offset": 0}
response = requests.get(url, headers=headers, params=params,timeout=30)
response.raise_for_status()
print("status code: ",response.status_code)
print(response.text)

status code:  200
{
    "error": false,
    "category": "Dark",
    "type": "twopart",
    "setup": "What's the most notable similarity between a pedophile and a roller coaster?",
    "delivery": "They both make a child scream, but the first ride's never the last",
    "flags": {
        "nsfw": true,
        "religious": false,
        "political": false,
        "racist": false,
        "sexist": false,
        "explicit": true
    },
    "id": 108,
    "safe": false,
    "lang": "en"
}


In [27]:
# offset based pagination

def fetch_all_offset(base_url: str, headers: dict, page_size: int = 100) -> list[dict]:
    """Fetch all records using offset-based pagination.

    Works for: Shopify, many internal APIs, older REST APIs.
    Limitation: if new records are inserted during extraction,
    you may get duplicates or miss records. For large datasets,
    prefer cursor-based pagination.
    """
    all_records = []
    offset = 0

    while True:
        response = requests.get(
            base_url,
            headers=headers,
            params={"limit": page_size, "offset": offset},
            timeout=30,
        )
        response.raise_for_status()
        data = response.json()

        records = data.get("results", [])
        if not records:
            break

        all_records.extend(records)
        offset += page_size

    return all_records


In [28]:
## exponential backoff

import time
import random
import requests
from requests.exceptions import HTTPError

def request_with_retry(
    method: str,
    url: str,
    max_retries: int = 5,
    base_delay: float = 1.0,
    max_delay: float = 60.0,
    **kwargs,
) -> requests.Response:
    """Make an HTTP request with exponential backoff and jitter.

    Retries on: 429 (rate limited), 500, 502, 503, 504 (server errors).
    Does NOT retry on: 400, 401, 403, 404 (client errors — these won't fix themselves).
    """
    kwargs.setdefault("timeout", 30)
    retryable_status_codes = {429, 500, 502, 503, 504}

    for attempt in range(max_retries + 1):
        try:
            response = requests.request(method, url, **kwargs)

            if response.status_code not in retryable_status_codes:
                response.raise_for_status()
                return response

            if attempt == max_retries:
                response.raise_for_status()  # Final attempt — raise the error

            # Check for Retry-After header (some APIs tell you exactly how long to wait)
            retry_after = response.headers.get("Retry-After")
            if retry_after:
                delay = float(retry_after)
            else:
                # Exponential backoff with full jitter
                delay = min(base_delay * (2 ** attempt), max_delay)
                delay = random.uniform(0, delay)

            print(f"Attempt {attempt + 1} got {response.status_code}. "
                  f"Retrying in {delay:.1f}s...")
            time.sleep(delay)

        except requests.exceptions.ConnectionError:
            if attempt == max_retries:
                raise
            delay = min(base_delay * (2 ** attempt), max_delay)
            time.sleep(random.uniform(0, delay))

    # Should not reach here, but just in case
    raise RuntimeError(f"Failed after {max_retries} retries: {url}")
